# NYC Parks & Recreation (DOPR) — Budget vs. Actual Spending

**Project:** NYC Budget Allocation Analysis & Recommendation System
**Agency scope:** Department of Parks and Recreation (DOPR)
**Covers Jira:** KAN-52 (EDA) · KAN-32 (DOPR insights — what's volatile, what stands out) · KAN-19 (Regression model for DOPR)

---

### What this notebook does

1. **Loads and audits** `ParksBudget_vs_Spending.csv` (FY2017–FY2027, budget-code level)
2. **Cleans** the panel — including two data traps that will silently wreck the analysis if missed
3. **EDA** — adopted vs. modified vs. actual, execution rates, where the money actually goes
4. **Volatility analysis** — which budget lines are stable and which are chronically mis-planned
5. **Regression models** — predict actual spending from budget features, with a proper time-based split
6. **FY2027 forecast** and CSV exports for the Tableau dashboard

### Two findings that shape everything downstream

Both were discovered during the audit in Section 2 — don't skip it:

- **FY2027 is a partial year.** Only ~$212M of a $1.42B budget is recorded as spent. NYC's fiscal year 2027 runs Jul 2026–Jun 2027, so at the time this data was pulled the year had barely started. Including FY2027 in any execution-rate or accuracy calculation makes Parks look catastrophically underspent. It is held out as a **prediction target**, never as training or evaluation data.
- **65% of rows are entirely zero.** 7,733 of 11,787 rows have zero in every money column — inactive/reserved budget codes. Leaving them in drags every mean toward zero and inflates R² by making the model good at predicting "nothing."


## 1. Setup and load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

CSV_PATH = "ParksBudget_vs_Spending.csv"   # <- change if your file lives elsewhere

# CRITICAL: Budget Code must be read as a string.
# It contains leading zeros ("0110") and alphanumeric codes ("IMP2", "HTS2", "CV06").
# Default inference mangles them and silently merges distinct codes.
df = pd.read_csv(CSV_PATH, dtype={"Budget Code": str})
df = df.rename(columns={"Budget Code": "Budget_Code"})

MONEY_COLS = ["Adopted_Budget", "Modified_Budget", "Post_Adjustments",
              "Pre_Encumbered", "Actual_Spending"]

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Fiscal years: {df.Year.min()}-{df.Year.max()}")
print(f"Unique budget codes: {df.Budget_Code.nunique():,}")
df.head()

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print(f"Duplicate (Year, Budget_Code) pairs: {df.duplicated(['Year','Budget_Code']).sum()}")

## 2. Data quality audit

This is the section that earns its keep. Four checks, each of which changes how the rest of the notebook is built.

### 2.1 The zero-row problem

A budget code exists in the schema for every year whether or not it was funded. Most weren't.

In [ ]:
all_zero = (df[MONEY_COLS] == 0).all(axis=1)

print(f"Rows where every money column is 0 : {all_zero.sum():,}  ({all_zero.mean():.1%})")
print(f"Rows with any financial activity    : {(~all_zero).sum():,}")
print()

# How many codes are actually live in a given year?
live_per_year = df[~all_zero].groupby("Year").size()
print("Active budget codes per fiscal year:")
print(live_per_year.to_string())

**Takeaway:** Parks operates roughly **370 live budget lines per year**, not 1,179. The remaining rows are
schema placeholders. Keeping them would mean a model that scores well by learning "predict $0," which is
true 65% of the time and useless for budget planning.

### 2.2 The FY2027 partial-year trap

Compare total budget against total recorded spending, by year.

In [ ]:
yearly = df.groupby("Year").agg(
    Adopted=("Adopted_Budget", "sum"),
    Modified=("Modified_Budget", "sum"),
    Actual=("Actual_Spending", "sum"),
).div(1e9)
yearly["Spend_vs_Modified"] = yearly.Actual / yearly.Modified

print("All figures in $ billions\n")
print(yearly.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(yearly.index, yearly.Spend_vs_Modified,
       color=["#4C72B0"] * (len(yearly) - 1) + ["#C44E52"])
ax.axhline(1.0, color="black", ls="--", lw=1, label="Full budget execution")
ax.set(title="Share of modified budget actually spent, by fiscal year",
       xlabel="Fiscal year", ylabel="Actual / Modified")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.annotate("FY2027 incomplete —\nyear still in progress", xy=(2027, 0.148),
            xytext=(2024.3, 0.45), arrowprops=dict(arrowstyle="->", color="#C44E52"),
            color="#C44E52", fontweight="bold")
ax.legend()
plt.tight_layout(); plt.show()

FY2017–FY2026 all land between roughly **85% and 95%** execution — a normal, stable range for a large
operating agency. FY2027 sits at **~15%**, which is not an efficiency collapse; it is a fiscal year that had
only just begun when this extract was taken.

**Decision:** `CLOSED_YEAR_MAX = 2026`. Everything descriptive and every model evaluation uses FY2017–FY2026.
FY2027 is carried separately and used only as the forecast target in Section 8.

### 2.3 Negative values

Not errors — these are offsetting adjustments and intra-city transfers, and they are legitimate accounting.

In [ ]:
neg = df[(df[MONEY_COLS] < 0).any(axis=1)]
print(f"Rows containing at least one negative amount: {len(neg)}")
print()
print(neg[["Year", "Budget_Code"] + MONEY_COLS]
      .sort_values("Modified_Budget").head(10).to_string(index=False))

These are kept in the descriptive analysis (they are real dollars) but the target variable is clipped at
zero for modeling, since "negative spending" is not a meaningful thing to forecast.

### 2.4 Budget code structure

The first character of the code carries information — borough and program-type prefixes.

In [ ]:
codes = df.Budget_Code
print(f"All codes are 4 characters: {(codes.str.len() == 4).all()}")
print(f"Purely numeric codes : {codes.str.isdigit().sum():,}")
print(f"Alphanumeric codes   : {(~codes.str.isdigit()).sum():,}")
print()
print("Most common first characters among ACTIVE rows:")
print(df[~all_zero].Budget_Code.str[0].value_counts().head(12).to_string())
print()
print("Sample alphanumeric codes (these encode program type — CV = COVID-era, IMP = improvements, etc.):")
print(sorted(set(codes[~codes.str.isdigit()]))[:24])

## 3. Build the analysis frame

In [ ]:
CLOSED_YEAR_MAX = 2026     # last completed fiscal year
FORECAST_YEAR   = 2027     # in-progress year, held out for prediction

# Drop schema placeholders: keep any row with financial activity
active = df[(df[MONEY_COLS] != 0).any(axis=1)].copy()

# Closed-year frame for all descriptive analysis
hist = active[active.Year <= CLOSED_YEAR_MAX].copy()

# Core derived measures
hist["Variance"]        = hist.Actual_Spending - hist.Modified_Budget      # + = overspend
hist["Budget_Growth"]   = hist.Modified_Budget - hist.Adopted_Budget       # mid-year adjustment
hist["Underspend"]      = hist.Modified_Budget - hist.Actual_Spending

# Execution rate: guard against a tiny denominator producing absurd ratios.
# Without the $10k floor, a code with a $500 budget and $1.4M spend reports 2,773x.
MIN_DENOM = 10_000
hist["Exec_Rate"] = np.where(hist.Modified_Budget > MIN_DENOM,
                             hist.Actual_Spending / hist.Modified_Budget, np.nan)

print(f"Active rows              : {len(active):,}")
print(f"Closed-year rows (<={CLOSED_YEAR_MAX}) : {len(hist):,}")
print(f"Rows with valid Exec_Rate: {hist.Exec_Rate.notna().sum():,}")
print()
print("Execution rate distribution (closed years only):")
print(hist.Exec_Rate.describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

The 95th percentile sits well above 1.0 — a meaningful share of lines overspend their modified budget.
That is the central tension of this project and Section 5 quantifies it.

## 4. Exploratory analysis (KAN-52 / KAN-14)

### 4.1 The three budget stages over time

In [ ]:
stage = hist.groupby("Year")[["Adopted_Budget", "Modified_Budget", "Actual_Spending"]].sum().div(1e9)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(stage.index, stage.Adopted_Budget,  "o-", lw=2, label="Adopted budget")
ax.plot(stage.index, stage.Modified_Budget, "s-", lw=2, label="Modified budget")
ax.plot(stage.index, stage.Actual_Spending, "^-", lw=2, label="Actual spending")
ax.fill_between(stage.index, stage.Actual_Spending, stage.Modified_Budget,
                alpha=0.15, color="red", label="Unspent")
ax.set(title="NYC Parks: adopted vs. modified vs. actual, FY2017–FY2026",
       xlabel="Fiscal year", ylabel="$ billions")
ax.legend()
plt.tight_layout(); plt.show()

stage["Unspent"]     = stage.Modified_Budget - stage.Actual_Spending
stage["Mid_Yr_Adj"]  = stage.Modified_Budget - stage.Adopted_Budget
print(stage.round(3).to_string())

**What stands out:**

- Parks runs a **$1.0–1.3B** operating envelope that grew about **28%** across the decade.
- The modified budget nearly always lands **above** adopted — the agency reliably receives mid-year additions.
- A persistent **$50–150M annual gap** between modified budget and actual spending. This is the funding-gap
  signal the project proposal set out to find, and it is systematic rather than a one-off.
- **FY2020–FY2021** shows the pandemic dip: adopted budget was cut mid-year, the one period where modified
  fell *below* adopted.

### 4.2 Distribution of spending across budget codes

In [ ]:
latest = hist[hist.Year == CLOSED_YEAR_MAX].sort_values("Actual_Spending", ascending=False)
latest_nz = latest[latest.Actual_Spending > 0]

share = latest_nz.Actual_Spending.cumsum() / latest_nz.Actual_Spending.sum()
n_for_80 = int((share <= 0.80).sum() + 1)

print(f"FY{CLOSED_YEAR_MAX}: {len(latest_nz)} codes recorded spending")
print(f"{n_for_80} codes ({n_for_80/len(latest_nz):.1%}) account for 80% of all dollars spent")
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(range(1, len(share) + 1), share.values, lw=2, color="#4C72B0")
axes[0].axhline(0.8, color="red", ls="--", lw=1)
axes[0].axvline(n_for_80, color="red", ls="--", lw=1)
axes[0].set(title="Spending concentration (Pareto)", xlabel="Budget codes, ranked",
            ylabel="Cumulative share of spending")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

top10 = latest_nz.head(10).iloc[::-1]
axes[1].barh(top10.Budget_Code, top10.Actual_Spending / 1e6, color="#55A868")
axes[1].set(title=f"Top 10 budget codes by spend, FY{CLOSED_YEAR_MAX}", xlabel="$ millions")
plt.tight_layout(); plt.show()

print(latest_nz.head(10)[["Budget_Code", "Adopted_Budget", "Modified_Budget",
                          "Actual_Spending", "Exec_Rate"]].to_string(index=False))

Spending is **heavily concentrated**: a small minority of codes carries most of the budget while hundreds
of lines are under $100k. This is why the modeling section reports dollar-weighted error alongside per-line
error — getting one $92M line wrong outweighs getting fifty $50k lines right.

### 4.3 Execution rate distribution

In [ ]:
valid = hist.Exec_Rate.dropna()
clipped = valid.clip(0, 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(clipped, bins=50, color="#4C72B0", edgecolor="white")
axes[0].axvline(1.0, color="red", ls="--", lw=2, label="Budget fully spent")
axes[0].set(title="Execution rate distribution (clipped at 2.0)",
            xlabel="Actual / Modified budget", ylabel="Number of code-years")
axes[0].legend()

by_year = hist.groupby("Year").Exec_Rate.median()
axes[1].plot(by_year.index, by_year.values, "o-", lw=2, color="#C44E52")
axes[1].axhline(1.0, color="black", ls="--", lw=1)
axes[1].set(title="Median execution rate by year", xlabel="Fiscal year",
            ylabel="Median actual / modified")
plt.tight_layout(); plt.show()

buckets = pd.cut(valid, [-np.inf, 0.5, 0.9, 1.1, np.inf],
                 labels=["Severely underspent (<50%)", "Underspent (50-90%)",
                         "On target (90-110%)", "Overspent (>110%)"])
print("Where code-years land:")
print(buckets.value_counts().reindex(buckets.cat.categories).to_string())
print()
print(f"Median execution rate: {valid.median():.1%}")

**This is the headline EDA finding for KAN-32.** The distribution is *wide and flat*, not clustered around
100%. Only **19%** of code-years land within 10% of their modified budget. Nearly 30% come in under half their
budget, and 22% overspend by more than 10%. Median execution is **79.5%**.

So the agency total looks well-controlled (88–95% every year) while the line-level picture is chaotic — the
errors are large in both directions and largely offset in aggregate. Any recommendation system built on this
data has to work at the line level, because the aggregate hides the problem.

## 5. Volatility analysis (KAN-32: what's volatile, what stands out)

### 5.1 Which codes are chronically mis-planned?

Only codes with a meaningful, sustained presence are considered — at least 5 years of history and $1M average
budget — so that a single noisy small line doesn't top the chart.

In [ ]:
MIN_YEARS, MIN_AVG_BUDGET = 5, 1_000_000

prof = (hist.groupby("Budget_Code")
            .agg(Years=("Year", "nunique"),
                 Avg_Modified=("Modified_Budget", "mean"),
                 Avg_Actual=("Actual_Spending", "mean"),
                 Std_Actual=("Actual_Spending", "std"),
                 Mean_Exec=("Exec_Rate", "mean"),
                 Std_Exec=("Exec_Rate", "std"),
                 Total_Variance=("Variance", "sum"))
            .query("Years >= @MIN_YEARS and Avg_Modified >= @MIN_AVG_BUDGET"))

# Coefficient of variation: volatility normalised by size, so a $90M line
# and a $2M line can be ranked on the same scale.
prof["CV_Spending"] = prof.Std_Actual / prof.Avg_Actual.replace(0, np.nan)

print(f"Codes meeting the stability filter: {len(prof)}")
print()
print("=== MOST VOLATILE (highest coefficient of variation) ===")
print(prof.nlargest(10, "CV_Spending")[
    ["Years", "Avg_Modified", "Avg_Actual", "CV_Spending", "Mean_Exec"]].to_string())
print()
print("=== MOST STABLE (lowest coefficient of variation) ===")
print(prof.nsmallest(10, "CV_Spending")[
    ["Years", "Avg_Modified", "Avg_Actual", "CV_Spending", "Mean_Exec"]].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(prof.Avg_Modified / 1e6, prof.CV_Spending,
                s=40, alpha=0.6, c=prof.Mean_Exec.clip(0, 1.5), cmap="RdYlGn_r",
                edgecolor="grey", linewidth=0.4)
ax.set_xscale("log")
ax.set(title="Budget line size vs. spending volatility",
       xlabel="Average modified budget, $M (log scale)",
       ylabel="Coefficient of variation of actual spending")
plt.colorbar(sc, label="Mean execution rate")

for code_id, row in prof.nlargest(5, "CV_Spending").iterrows():
    ax.annotate(code_id, (row.Avg_Modified / 1e6, row.CV_Spending),
                fontsize=9, fontweight="bold",
                xytext=(5, 4), textcoords="offset points")
plt.tight_layout(); plt.show()

**Pattern:** volatility generally falls as line size rises — the biggest operational codes (`3808`, `2320`)
have coefficients of variation near **0.03–0.06**, meaning they spend almost the same amount every year.

But note the exceptions in the volatile list. Several codes (`2660`, `6803`, `0739`) carry multi-million-dollar
modified budgets and spend **almost nothing** — mean execution rates of 0–5%. These aren't noisy lines; they're
lines that hold money and never disburse it. And `CRC2` is a $26.5M line with a CV above 2.0, so size alone
doesn't guarantee stability.

Useful distinction for the recommendation system: *large steady operations lines can be forecast mechanically;
persistent zero-spend lines are a reallocation question, not a forecasting one.*

### 5.2 Biggest chronic overspenders and underspenders

In [ ]:
print("=== CHRONIC OVERSPENDERS (largest cumulative spend above modified budget) ===")
print(prof.nlargest(8, "Total_Variance")[
    ["Years", "Avg_Modified", "Avg_Actual", "Mean_Exec", "Total_Variance"]].to_string())
print()
print("=== CHRONIC UNDERSPENDERS (largest cumulative unspent balance) ===")
print(prof.nsmallest(8, "Total_Variance")[
    ["Years", "Avg_Modified", "Avg_Actual", "Mean_Exec", "Total_Variance"]].to_string())

In [ ]:
top_var = pd.concat([prof.nlargest(8, "Total_Variance"), prof.nsmallest(8, "Total_Variance")])
top_var = top_var.sort_values("Total_Variance") / 1

fig, ax = plt.subplots(figsize=(11, 6.5))
colors = ["#C44E52" if v < 0 else "#55A868" for v in top_var.Total_Variance]
ax.barh(top_var.index, top_var.Total_Variance / 1e6, color=colors)
ax.axvline(0, color="black", lw=1)
ax.set(title="Cumulative variance by budget code, FY2017–FY2026\n(green = spent above budget, red = left unspent)",
       xlabel="Cumulative actual − modified, $ millions", ylabel="Budget code")
plt.tight_layout(); plt.show()

### 5.3 Year-over-year volatility of the agency total

In [ ]:
tot = hist.groupby("Year")[["Adopted_Budget", "Modified_Budget", "Actual_Spending"]].sum()
yoy = tot.pct_change().mul(100).dropna()

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(yoy)); w = 0.27
ax.bar(x - w, yoy.Adopted_Budget,  w, label="Adopted")
ax.bar(x,     yoy.Modified_Budget, w, label="Modified")
ax.bar(x + w, yoy.Actual_Spending, w, label="Actual")
ax.axhline(0, color="black", lw=1)
ax.set(title="Year-over-year growth rates", xlabel="Fiscal year", ylabel="% change")
ax.set_xticks(x); ax.set_xticklabels(yoy.index)
ax.legend()
plt.tight_layout(); plt.show()

print(yoy.round(2).to_string())
print()
print("Volatility (std dev of YoY growth):")
print(yoy.std().round(2).to_string())

**Actual spending is the smoothest of the three series.** The adopted budget swings hardest year to year —
it is a political and planning artifact, while spending is anchored by payroll and fixed operating costs. This
has a direct modeling consequence: *last year's actual spending should be a strong predictor, and the adopted
budget alone should be a weak one.* Section 7 confirms exactly that.

## 6. Feature engineering

Building a **panel** with lagged features. Every feature uses only information available *before* the target
year — no peeking at the outcome.

In [ ]:
panel = active.sort_values(["Budget_Code", "Year"]).copy()
grp = panel.groupby("Budget_Code")

# --- Lagged history (shift(1) = prior fiscal year for the same code) ---
panel["Prior_Actual"]   = grp.Actual_Spending.shift(1)
panel["Prior_Modified"] = grp.Modified_Budget.shift(1)
panel["Prior_Adopted"]  = grp.Adopted_Budget.shift(1)
panel["Prior_Exec"]     = np.where(panel.Prior_Modified > MIN_DENOM,
                                   panel.Prior_Actual / panel.Prior_Modified, np.nan)

# --- Rolling history (shift first, then roll, so the current year is excluded) ---
panel["Actual_3yr_Mean"] = grp.Actual_Spending.transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean())
panel["Actual_3yr_Std"]  = grp.Actual_Spending.transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).std())

# --- Code characteristics ---
panel["Years_Active"]   = grp.cumcount()
panel["Adopted_Growth"] = np.where(panel.Prior_Adopted > MIN_DENOM,
                                   panel.Adopted_Budget / panel.Prior_Adopted - 1, 0)
panel["Is_Alpha"]       = panel.Budget_Code.str[0].str.isalpha().astype(int)

# --- Imputation ---
# 0 is meaningful here: a first-year code genuinely has no prior spending.
for col in ["Prior_Actual", "Prior_Modified", "Prior_Adopted",
            "Actual_3yr_Mean", "Actual_3yr_Std"]:
    panel[col] = panel[col].fillna(0)

panel["Prior_Exec"]     = panel.Prior_Exec.fillna(panel.Prior_Exec.median()).clip(0, 3)
panel["Adopted_Growth"] = panel.Adopted_Growth.replace([np.inf, -np.inf], 0).clip(-5, 5)

# --- Program-type prefix, one-hot (keep only well-populated prefixes) ---
prefix_dummies = pd.get_dummies(panel.Budget_Code.str[0], prefix="pfx")
prefix_cols = list(prefix_dummies.columns[prefix_dummies.sum() >= 30])
panel = pd.concat([panel, prefix_dummies[prefix_cols]], axis=1)

print(f"Panel shape: {panel.shape}")
print(f"Prefix features retained: {prefix_cols}")
panel[["Year", "Budget_Code", "Adopted_Budget", "Modified_Budget", "Prior_Actual",
       "Prior_Exec", "Actual_3yr_Mean", "Years_Active"]].head(8)

### Two feature regimes

`Modified_Budget` is only finalised late in the fiscal year. A model that uses it isn't *forecasting* — it's
**nowcasting**. Both are useful, but they answer different questions, so they're kept separate and compared.

In [ ]:
PLANNING_FEATURES = [
    "Adopted_Budget", "Prior_Actual", "Prior_Modified", "Prior_Adopted", "Prior_Exec",
    "Actual_3yr_Mean", "Actual_3yr_Std", "Years_Active", "Adopted_Growth", "Is_Alpha",
] + prefix_cols

INYEAR_FEATURES = PLANNING_FEATURES + [
    "Modified_Budget", "Post_Adjustments", "Pre_Encumbered",
]

TARGET = "Actual_Spending"

print(f"Planning-time features (available at budget adoption): {len(PLANNING_FEATURES)}")
print(f"In-year features (adds mid-year revisions)           : {len(INYEAR_FEATURES)}")

### Time-based train / validation / test split

A random split would let the model see FY2026 while predicting FY2022 — **temporal leakage**, and the reported
accuracy would be fiction. Budget forecasting is inherently a forward-in-time problem, so the split respects
chronology.

FY2017 is dropped from modeling because it has no prior year to lag from.

In [ ]:
model_df = panel[(panel.Year >= 2018) & (panel.Year <= CLOSED_YEAR_MAX)].copy()
model_df[TARGET] = model_df[TARGET].clip(lower=0)

train = model_df[model_df.Year <= 2024]
val   = model_df[model_df.Year == 2025]
test  = model_df[model_df.Year == 2026]

print(f"Train  FY2018-FY2024 : {len(train):>5,} rows   ${train[TARGET].sum()/1e9:.2f}B")
print(f"Val    FY2025        : {len(val):>5,} rows   ${val[TARGET].sum()/1e9:.2f}B")
print(f"Test   FY2026        : {len(test):>5,} rows   ${test[TARGET].sum()/1e9:.2f}B")

## 7. Regression models (KAN-19)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def score(y_true, y_pred, label, split):
    """Per-line accuracy AND dollar-weighted accuracy.

    Total_Error_% matters as much as R2 here: a budget office cares whether the
    agency-level forecast is right, not just whether individual lines correlate.
    """
    y_pred = np.clip(y_pred, 0, None)
    return {
        "Model": label,
        "Split": split,
        "MAE_$M": mean_absolute_error(y_true, y_pred) / 1e6,
        "RMSE_$M": np.sqrt(mean_squared_error(y_true, y_pred)) / 1e6,
        "R2": r2_score(y_true, y_pred),
        "Total_Error_%": 100 * (y_pred.sum() - y_true.sum()) / y_true.sum(),
    }

### 7.1 Naive baselines

Any ML model has to beat these to justify itself. Skipping this step is the most common way a project
overstates its own results.

In [ ]:
baseline_rows = []
for split_name, d in [("FY2025 (val)", val), ("FY2026 (test)", test)]:
    baseline_rows += [
        score(d[TARGET], d.Prior_Actual,                 "Baseline: repeat last year", split_name),
        score(d[TARGET], d.Adopted_Budget.clip(lower=0), "Baseline: adopted budget",   split_name),
        score(d[TARGET], d.Modified_Budget.clip(lower=0),"Baseline: modified budget",  split_name),
    ]

baselines = pd.DataFrame(baseline_rows)
print(baselines.round(3).to_string(index=False))

**No single baseline dominates, and that's the interesting part.**

- *Repeat last year* has the **lowest MAE** ($0.75–0.91M) but unstable R² — 0.66 on FY2025, 0.87 on FY2026. It
  nails the many small stable lines and misses badly when a line changes.
- *Adopted budget* and *modified budget* have **better R²** (0.82–0.88) but worse MAE. They track the large
  lines well and are noisy on small ones.
- Both budget baselines **overshoot the total** on FY2026 (+6.1% and +11.9%) — the agency does not spend what
  it is given, which is exactly the funding gap Section 4.1 showed.

Any model claimed as useful has to beat a moving target: MAE below ~$0.75M *and* R² above ~0.88.

### 7.2 Model training

One implementation note worth flagging. Spending spans $0 to $122M, so log-transforming the target is the
instinctive move. It was tested and **rejected**: back-transforming with `expm1` introduces retransformation
bias that understated the agency total by 11–13%, and Duan's smearing correction overcorrected badly because
of the mass of near-zero lines. Tree ensembles don't need the target normalised, so they are trained directly
on dollars. The choice is empirical, not stylistic — the comparison is left in below.

In [ ]:
def build_models():
    return {
        "Ridge regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "Random Forest": RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "Gradient Boosting": HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
            l2_regularization=1.0, random_state=42),
    }


results, fitted = [], {}

for regime, feats in [("Planning-time", PLANNING_FEATURES), ("In-year", INYEAR_FEATURES)]:
    X_train, y_train = train[feats].astype(float), train[TARGET]
    for name, model in build_models().items():
        model.fit(X_train, y_train)
        fitted[(regime, name)] = (model, feats)
        for split_name, d in [("FY2025 (val)", val), ("FY2026 (test)", test)]:
            r = score(d[TARGET], model.predict(d[feats].astype(float)),
                      f"{regime} — {name}", split_name)
            results.append(r)

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

In [ ]:
# The log-target variant that was rejected — shown for transparency
X_train = train[INYEAR_FEATURES].astype(float)
log_rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                               random_state=42, n_jobs=-1)
log_rf.fit(X_train, np.log1p(train[TARGET]))

log_rows = [score(d[TARGET], np.expm1(log_rf.predict(d[INYEAR_FEATURES].astype(float))),
                  "In-year — RF (log target)", s)
            for s, d in [("FY2025 (val)", val), ("FY2026 (test)", test)]]

print("Log-target variant — note the Total_Error_% column:")
print(pd.DataFrame(log_rows).round(3).to_string(index=False))

### 7.3 Model comparison

In [ ]:
comparison = pd.concat([baselines, results_df], ignore_index=True)
test_only = comparison[comparison.Split == "FY2026 (test)"].sort_values("R2", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
y = np.arange(len(test_only))
colors = ["#8C8C8C" if m.startswith("Baseline") else "#4C72B0" for m in test_only.Model]

axes[0].barh(y, test_only.R2, color=colors)
axes[0].set_yticks(y); axes[0].set_yticklabels(test_only.Model, fontsize=9)
axes[0].set(title="R² on held-out FY2026", xlabel="R²", xlim=(0, 1))

axes[1].barh(y, test_only["MAE_$M"], color=colors)
axes[1].set_yticks(y); axes[1].set_yticklabels([])
axes[1].set(title="Mean absolute error on FY2026", xlabel="MAE, $ millions")
plt.tight_layout(); plt.show()

best_key = max(fitted, key=lambda k: results_df[
    (results_df.Model == f"{k[0]} — {k[1]}") & (results_df.Split == "FY2026 (test)")].R2.iloc[0])
best_model, best_feats = fitted[best_key]
print(f"Best model: {best_key[0]} — {best_key[1]}")

**Reading the results honestly — two of these contradict what I expected going in:**

- **Every model beats every baseline on R²** (0.89–0.96 vs. 0.66–0.88), and the gain is real. But on **MAE the
  win is narrow**: the best model lands at $0.78M on FY2026 against $0.75M for simply repeating last year's
  actual. The models earn their keep on the *large* lines, where R² and RMSE are decided, not on typical ones.
  Report both numbers — quoting only R² would overstate the result.

- **Ridge does fine here** (R² 0.90–0.94), which is *not* what happened in an earlier version of this notebook
  that fed it log-transformed features — that variant produced wild negative R². The difference is
  `StandardScaler`. On raw dollar features spanning six orders of magnitude the design matrix is
  ill-conditioned and the coefficients blow up. Scaled, the linear model is perfectly competitive. Worth
  knowing before anyone claims "linear models don't work on budget data."

- **Planning-time ≈ in-year**, and planning-time Random Forest is actually the *best* model on FY2026 test
  (R² 0.94). This was the surprise. Knowing the mid-year modified budget adds almost nothing once you have the
  adopted budget plus spending history — which is genuinely good news operationally, because the planning-time
  model can be run **before the fiscal year begins**, when the recommendation is still actionable.

### 7.4 Feature importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = (pd.Series(best_model.feature_importances_, index=best_feats)
             .sort_values(ascending=False).head(12))
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(imp.index[::-1], imp.values[::-1], color="#55A868")
    ax.set(title=f"Top features — {best_key[0]} {best_key[1]}", xlabel="Importance")
    plt.tight_layout(); plt.show()
    print(imp.round(4).to_string())
else:
    print("Selected model does not expose feature_importances_.")

Two things to carry into the write-up:

- **`Adopted_Budget` (~0.51) and `Prior_Actual` (~0.29) carry 80% of the signal.** Everything else is
  rounding. The engineered rolling features add little beyond the simple lag, and `Prior_Exec` — which seemed
  like it should matter a lot — contributes almost nothing.
- **The prefix dummies contribute essentially zero.** Program-type is not predictive once size and history are
  known. Report this as a negative result rather than quietly dropping it; it tells the team not to invest
  further in categorical encoding of budget codes.

### 7.5 Error analysis — where the model actually fails

In [ ]:
err = test.copy()
err["Predicted"] = np.clip(best_model.predict(test[best_feats].astype(float)), 0, None)
err["Error"] = err.Predicted - err[TARGET]
err["Abs_Error"] = err.Error.abs()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

nz = err[err[TARGET] > 0]
axes[0].scatter(nz[TARGET] / 1e6, nz.Predicted / 1e6, alpha=0.5, s=28, color="#4C72B0")
lim = max(nz[TARGET].max(), nz.Predicted.max()) / 1e6
axes[0].plot([0, lim], [0, lim], "r--", lw=1.5, label="Perfect prediction")
axes[0].set(xscale="log", yscale="log", title="Predicted vs. actual, FY2026",
            xlabel="Actual, $M (log)", ylabel="Predicted, $M (log)")
axes[0].legend()

size_bin = pd.cut(err[TARGET], [-0.01, 1e5, 1e6, 1e7, np.inf],
                  labels=["<$100k", "$100k-$1M", "$1M-$10M", ">$10M"])
by_size = err.groupby(size_bin, observed=True).agg(
    n=("Abs_Error", "size"), Mean_Abs_Err=("Abs_Error", "mean"), Total_Err=("Error", "sum"))
axes[1].bar(by_size.index.astype(str), by_size.Mean_Abs_Err / 1e6, color="#C44E52")
axes[1].set(title="Mean absolute error by line size", xlabel="Actual spending bracket",
            ylabel="Mean abs. error, $M")
plt.tight_layout(); plt.show()

print(by_size.round(2).to_string())
print()
print("10 largest prediction errors:")
print(err.nlargest(10, "Abs_Error")[
    ["Budget_Code", "Adopted_Budget", "Modified_Budget", TARGET, "Predicted", "Error"]
].to_string(index=False))

Error concentrates almost entirely in the **>$10M** bracket. Small lines are predicted well in absolute
dollars simply because they're small. The practical implication for a recommendation system: **flag the top
~40 codes for analyst review** and let the model handle the long tail automatically.

## 8. FY2027 forecast

The model now predicts the in-progress year. FY2027 was never used for training or evaluation, so this is a
genuine out-of-sample forecast.

Only the **planning-time** model is usable here — FY2027's modified budget is not yet final.

In [ ]:
plan_key = ("Planning-time", best_key[1])
plan_model, plan_feats = fitted[plan_key]

fc = panel[panel.Year == FORECAST_YEAR].copy()
fc["Predicted_Spending"] = np.clip(plan_model.predict(fc[plan_feats].astype(float)), 0, None)
fc["Predicted_Exec_Rate"] = np.where(fc.Modified_Budget > MIN_DENOM,
                                     fc.Predicted_Spending / fc.Modified_Budget, np.nan)
fc["Projected_Unspent"] = fc.Modified_Budget - fc.Predicted_Spending

print(f"FY{FORECAST_YEAR} forecast using {plan_key[1]} (planning-time features)\n")
print(f"Adopted budget       : ${fc.Adopted_Budget.sum()/1e9:.3f}B")
print(f"Modified budget      : ${fc.Modified_Budget.sum()/1e9:.3f}B")
print(f"Predicted spending   : ${fc.Predicted_Spending.sum()/1e9:.3f}B")
print(f"Projected unspent    : ${fc.Projected_Unspent.sum()/1e6:.1f}M")
print(f"Implied execution    : {fc.Predicted_Spending.sum()/fc.Modified_Budget.sum():.1%}")
print(f"\n(Recorded so far in the partial year: ${fc.Actual_Spending.sum()/1e9:.3f}B — "
      "not comparable, year incomplete)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
hist_tot = hist.groupby("Year").Actual_Spending.sum() / 1e9

ax.plot(hist_tot.index, hist_tot.values, "o-", lw=2, label="Actual spending", color="#4C72B0")
ax.plot([CLOSED_YEAR_MAX, FORECAST_YEAR],
        [hist_tot.iloc[-1], fc.Predicted_Spending.sum() / 1e9],
        "s--", lw=2, color="#C44E52", label=f"FY{FORECAST_YEAR} forecast")
ax.scatter([FORECAST_YEAR], [fc.Modified_Budget.sum() / 1e9],
           marker="_", s=400, color="grey", label=f"FY{FORECAST_YEAR} modified budget")
ax.set(title="NYC Parks actual spending with FY2027 forecast",
       xlabel="Fiscal year", ylabel="$ billions")
ax.legend()
plt.tight_layout(); plt.show()

The model projects **~97% execution** for FY2027, above the FY2017–FY2026 range of 88–95%. Treat that as
optimistic: the model is trained on code-level patterns and doesn't know about hiring freezes, contract delays,
or the mid-year adjustments that historically leave $50–150M unspent. A reasonable write-up frames this as an
**upper bound** on FY2027 spending.

### Funding-priority ranking

A simple, defensible scoring rule for the recommendation system: lines projected to **exhaust or exceed** their
budget are candidates for increases; lines projected to leave large balances are candidates for reallocation.

In [ ]:
ranked = fc[fc.Modified_Budget > 1_000_000].copy()
ranked["Priority_Score"] = ranked.Predicted_Exec_Rate.fillna(0)

print("=== HIGHEST FUNDING URGENCY (projected to overrun budget) ===")
print(ranked.nlargest(10, "Priority_Score")[
    ["Budget_Code", "Adopted_Budget", "Modified_Budget",
     "Predicted_Spending", "Predicted_Exec_Rate"]].to_string(index=False))
print()
print("=== REALLOCATION CANDIDATES (large projected unspent balances) ===")
print(ranked.nlargest(10, "Projected_Unspent")[
    ["Budget_Code", "Modified_Budget", "Predicted_Spending",
     "Projected_Unspent", "Predicted_Exec_Rate"]].to_string(index=False))

**Two caveats to state in the write-up:**

1. The extreme predicted execution rates (10.9x, 6.6x) are not forecasting errors — codes like `2891` and
   `2295` have overspent their small nominal budgets every year for a decade, almost certainly because the
   actual funding is booked against a different line and charged here. They are **accounting artifacts, not
   underfunded programs**, and should be excluded or footnoted before this table goes in front of anyone.
2. This ranking is a *utilisation* signal, not a *need* signal. A line that underspends may be under-resourced
   in staff rather than over-budgeted, and the dataset cannot distinguish those. The proposal's goal of ranking
   by "funding urgency" needs service-demand data (park acreage, maintenance backlogs, usage counts) to be
   defensible — this is the budget-side half only.

## 9. Exports for the Tableau dashboard

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

# 1. Clean code-year fact table
hist[["Year", "Budget_Code", "Adopted_Budget", "Modified_Budget", "Post_Adjustments",
      "Pre_Encumbered", "Actual_Spending", "Variance", "Budget_Growth",
      "Underspend", "Exec_Rate"]].to_csv("outputs/dopr_clean_panel.csv", index=False)

# 2. Agency-level yearly summary
stage.reset_index().to_csv("outputs/dopr_yearly_summary.csv", index=False)

# 3. Per-code volatility profile
prof.reset_index().to_csv("outputs/dopr_code_volatility.csv", index=False)

# 4. Model comparison table
comparison.to_csv("outputs/dopr_model_results.csv", index=False)

# 5. FY2027 forecast + priority ranking
fc[["Year", "Budget_Code", "Adopted_Budget", "Modified_Budget", "Predicted_Spending",
    "Predicted_Exec_Rate", "Projected_Unspent"]].to_csv(
    "outputs/dopr_fy2027_forecast.csv", index=False)

for f in sorted(os.listdir("outputs")):
    print(f"  outputs/{f}  ({os.path.getsize('outputs/' + f):,} bytes)")

## 10. Summary of findings

**Data quality (must appear in the write-up):**

1. FY2027 is an incomplete fiscal year — ~15% of budget recorded as spent. Excluded from all descriptive and
   evaluation work; used only as a forecast target.
2. 65% of rows are inactive placeholder codes. Parks runs ~370 live budget lines per year, not 1,179.
3. `Budget Code` is alphanumeric with leading zeros and must be read as a string.
4. Execution rates need a denominator floor — tiny budgets otherwise produce ratios in the thousands.

**Spending patterns (KAN-52 / KAN-14):**

5. Parks' actual spending grew ~28% over FY2017–FY2026, from $1.02B to $1.30B; the adopted budget grew
   faster (~36%), so the plan-to-spend gap widened.
6. The modified budget almost always exceeds adopted — mid-year additions are routine, not exceptional.
7. A persistent $50–150M annual gap sits between modified budget and actual spending.
8. Spending is highly concentrated: a small minority of codes carries 80% of dollars.

**Volatility (KAN-32):**

9. Only **19%** of code-years land within 10% of their modified budget; median execution is 79.5%. Aggregate
   control (88–95% every year) masks large offsetting line-level errors.
10. Volatility generally falls with line size — the biggest operations codes have CV near 0.03–0.06 — but
    several multi-million-dollar lines (`2660`, `6803`, `0739`, `7000`) hold budget and spend near **zero**
    year after year. Code `7000` alone carries ~$37M/yr and has never recorded spending across the decade.
11. Actual spending is the *least* volatile of the three series (std dev of YoY growth 4.6%); the adopted
    budget is the most (10.4%).

**Modeling (KAN-19):**

12. Best model: **planning-time Random Forest**, R² 0.94 / MAE $0.78M on held-out FY2026. All models beat all
    baselines on R², but on MAE the margin over "repeat last year" is narrow ($0.78M vs $0.75M) — the models
    win on large lines, not typical ones. Quote both metrics.
13. Ridge is **competitive** (R² 0.90–0.94) once features are standardised. An earlier log-feature variant
    without scaling produced wildly negative R² — an ill-conditioning artifact, not evidence against linear
    models.
14. Knowing the mid-year modified budget adds **almost nothing** over adopted budget plus history, so the
    model can be run before the fiscal year starts — when the recommendation is still actionable.
15. Log-transforming the *target* introduced an 11–14% downward bias in the agency total and was rejected.
16. `Adopted_Budget` (~0.51) and `Prior_Actual` (~0.29) carry ~80% of feature importance. Program-type prefix
    contributes nothing.
17. Prediction error concentrates in the >$10M bracket (mean abs. error $4.7M across 26 lines) — auto-forecast
    the long tail, route large lines to analyst review.

**Next steps:**

- Run this same notebook against DOT and DOE for the cross-agency comparison (KAN-14)
- Bring in service-demand data (park acreage, maintenance backlogs, usage) — required before any "funding
  urgency" claim is defensible
- Add prediction intervals via quantile regression, so the recommendation system communicates uncertainty
  rather than a single point estimate
